In [ ]:
import numpy as np
import scipy as sp
import pandas as pd
import sklearn as skl
import matplotlib.pyplot as plt
import xgboost as xgb
import pickle
from xgboost.sklearn import XGBClassifier
from sklearn import cross_validation, metrics   #Additional scklearn functions
from sklearn.grid_search import GridSearchCV   #Perforing grid search
from sklearn.cross_validation import train_test_split, cross_val_score, KFold
from sklearn.metrics import log_loss, confusion_matrix, accuracy_score
from sklearn.preprocessing import minmax_scale


In [ ]:
# load training data
max_features = 1024
train = pd.read_csv('data/train.csv', header=None, sep=',', usecols = list(range(0,max_features)))
label = pd.read_csv("data/train_label.csv", sep=',')

In [ ]:
X = train
number_of_nan = X.isnull().sum(axis=1)
X = X.fillna(0)
X = X.astype(int)
number_valid = max_features - number_of_nan
X_describe = X.apply(pd.DataFrame.describe, axis=1)
X = pd.concat([X, number_of_nan, number_valid, X_describe], axis=1)
X.columns.values[1024] = "number_of_nan"
X.columns.values[1025] = "number_valid"

In [ ]:
y = label.iloc[:, -1]

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.05, random_state=42)

In [ ]:
model = XGBClassifier(
 learning_rate =0.1,
 n_estimators=2000,
 max_depth=8,
)

#model.fit(X, y, eval_set=[(X, y), (x_test, y_test)], eval_metric='logloss', verbose=True)
model.fit(x_train, y_train, eval_set=[(x_test, y_test)], eval_metric='logloss',early_stopping_rounds=500, verbose=True)

In [ ]:

pickle.dump(model, open("xgb.model.dat", "wb"))
#loaded_model = pickle.load(open("pima.pickle.dat", "rb"))

In [ ]:
test = pd.read_csv('data/test.csv', header=None, names = list(range(0,max_features)))

number_of_nan = test.isnull().sum(axis=1)
test = test.fillna(0)
test = test.astype(int)
number_valid = max_features - number_of_nan
test_describe = test.apply(pd.DataFrame.describe, axis=1)
test = pd.concat([test, number_of_nan, number_valid, test_describe], axis=1)
test.columns.values[1024] = "number_of_nan"
test.columns.values[1025] = "number_valid"

In [ ]:
best_iter = model.best_iteration
y_pred = model.predict_proba(test, ntree_limit = best_iter)[:, 1]

df = pd.DataFrame(y_pred)
df.to_csv("data/sgb_prob.csv")

In [ ]:
best_iter